# Práctica Lynis – Isolation Forest
**Celda 1:** lee tu `lynis.log` desde GitHub y lo convierte a CSV (crea el `df`).
**Celda 2:** código del profe (Isolation Forest).

In [ ]:
import re
import urllib.request
import numpy as np
import pandas as pd

USUARIO = "uazedrd-svg"
HOST = "edd-VMware-Virtual-Platform"
URL = f"https://raw.githubusercontent.com/{USUARIO}/lynis/main/logs/{HOST}/lynis.log"

texto = urllib.request.urlopen(URL).read().decode("utf-8", errors="replace")

# Cada línea del log empieza con fecha y hora: 2026-10-09 09:40:12 mensaje
patron = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (.*)$")
filas = []
for linea in texto.splitlines():
    m = patron.match(linea)
    if m:
        filas.append({"timestamp": m.group(1), "mensaje": m.group(2)})

df = pd.DataFrame(filas)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Columnas numéricas que usará Isolation Forest
df["segundos_desde_inicio"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds()
df["duracion_s"] = (df["timestamp"].shift(-1) - df["timestamp"]).dt.total_seconds().fillna(0)
df["longitud_mensaje"] = df["mensaje"].str.len()
df["es_warning"] = df["mensaje"].str.contains("warning", case=False).astype(int)
df["es_suggestion"] = df["mensaje"].str.contains("suggestion", case=False).astype(int)
df["es_test"] = df["mensaje"].str.contains("Performing test ID", case=False).astype(int)

df.to_csv("lynis_log.csv", index=False)
print("CSV generado: lynis_log.csv", df.shape)
df.head()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.exceptions import NotFittedError

print("Dimensión original del dataset:", df.shape)

result_cols = ["anomaly_score", "is_anomaly"]
features = df.drop(columns=result_cols, errors="ignore")

X = features.select_dtypes(include=[np.number]).copy()

X = X.replace([np.inf, -np.inf], np.nan)

if X.shape[1] == 0:
    raise ValueError(
        "No existen columnas numéricas para entrenar IsolationForest."
    )

X = X.dropna(axis=1, how="all")

if X.shape[1] == 0:
    raise ValueError("Todas las columnas numéricas están vacías.")

print("Características utilizadas:", X.columns.tolist())
print("Dimensión de X:", X.shape)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("isolation_forest", IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X)

df["anomaly_score"] = model.decision_function(X)
df["is_anomaly"] = model.predict(X)

df["anomaly_label"] = df["is_anomaly"].map({
    1: "Normal",
    -1: "Anomalía"
})

print("\nDistribución de resultados:")
print(df["anomaly_label"].value_counts())

print("\nPorcentaje de anomalías:")
print(
    df["anomaly_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nPrimeras observaciones:")
print(df.head(10).to_string(index=False))

df.to_csv("isolation_forest_resultados.csv", index=False)

print("\nArchivo generado: isolation_forest_resultados.csv")

**Extra:** las 10 líneas más anómalas (puntaje más bajo = más rara).

In [ ]:
df.sort_values("anomaly_score").head(10)[["timestamp", "duracion_s", "mensaje", "anomaly_score"]]

**Descargar los CSV** a tu computadora.

In [ ]:
from google.colab import files
files.download("lynis_log.csv")
files.download("isolation_forest_resultados.csv")